# Gold: per-case assessment

Joins **synthetic** cases to the variant reference and produces one assessment row per
reported variant, plus an explicit coverage statement per case.

| | |
| --- | --- |
| **Reads** | `silver_variant_reference`, `bronze_reference_coverage` |
| **Writes** | `gold_case_variant_assessment`, `gold_case_coverage`, `gold_case_summary` |

### Three states that must never collapse

| State | Meaning | What it is **not** |
| --- | --- | --- |
| `classified` | A submitted assertion exists in the release that was read | — |
| `no_reference_entry` | The variant is real but has no submission | Not benign |
| `gene_not_covered` | The gene was not on the panel, or could not be read | Not a negative result |

Only a variant classified benign is a negative finding. The other two are absences of
information, and the data model keeps them structurally apart so no downstream layer
can round them into reassurance.

**All patient data here is synthetic.** Variants referenced are real public ClinVar
records; the cases attached to them are fabricated for demonstration.

In [ ]:
PIPELINE_RUN_ID = ""

In [ ]:
import json
import uuid
from datetime import datetime, timezone

from pyspark.sql import functions as F

# --- cross-lakehouse reads -------------------------------------------------------
# spark.read.table() only resolves against this notebook's default lakehouse, so any
# table in a different layer is read by explicit OneLake path.
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table_name):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    path = (f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table_name}")
    return spark.read.format("delta").load(path)


from pyspark.sql.types import (
    BooleanType, IntegerType, StringType, StructField, StructType,
)

reference = lake_table("silver_lakehouse", "silver_variant_reference")
RUN_ID = PIPELINE_RUN_ID or (reference.orderBy(F.desc("generated_at_utc"))
                             .select("run_id").first()["run_id"])
GENERATED_AT = datetime.now(timezone.utc).isoformat()
reference = reference.filter(F.col("run_id") == RUN_ID).cache()
print("run_id:", RUN_ID, " reference variants:", reference.count())

## 1. Build synthetic cases against variants that actually came back

Accessions are selected from this run rather than hardcoded, so the demo holds whatever
the release contains. Each case is designed to exercise one of the three states.

In [ ]:
def pick(significance, gene=None, count=1):
    query = reference.filter(F.col("clinical_significance") == significance)
    if gene:
        query = query.filter(F.col("gene_symbol") == gene)
    rows = query.orderBy(F.desc("review_confidence"), "accession").limit(count).collect()
    return [row["accession"] for row in rows]


pathogenic = pick("Pathogenic") or pick("Likely pathogenic")
uncertain = pick("Uncertain significance")
benign = pick("Benign") or pick("Likely benign")

CASES = [
    {
        "case_id": "SYNTH-001",
        "referral_indication": "Infantile-onset seizures, developmental delay",
        "phenotype_terms": "HP:0001250 Seizure; HP:0001263 Global developmental delay",
        "panel_applied": "childhood_epilepsy_v1",
        "reported_variants": (pathogenic[:1] or []) + (benign[:1] or []),
        "uncalled_gene": None,
    },
    {
        "case_id": "SYNTH-002",
        "referral_indication": "Early-onset epileptic encephalopathy",
        "phenotype_terms": "HP:0200134 Epileptic encephalopathy",
        "panel_applied": "childhood_epilepsy_v1",
        "reported_variants": uncertain[:1] or [],
        "uncalled_gene": None,
    },
    {
        "case_id": "SYNTH-003",
        "referral_indication": "Febrile seizures, family history",
        "phenotype_terms": "HP:0002373 Febrile seizure",
        "panel_applied": "childhood_epilepsy_v1",
        # A plausible variant with no submission in the release. Deliberately not a real
        # accession: the point is what the pipeline does when the reference is silent.
        "reported_variants": ["VCV_NOT_SUBMITTED_0001"],
        "uncalled_gene": None,
    },
    {
        "case_id": "SYNTH-004",
        "referral_indication": "Seizures with tuberous sclerosis features",
        "phenotype_terms": "HP:0001250 Seizure; HP:0009721 Tuberous sclerosis",
        "panel_applied": "childhood_epilepsy_v1",
        "reported_variants": [],
        # TSC1/TSC2 were not on this panel. "No finding" here means nothing at all.
        "uncalled_gene": "TSC1",
    },
]

print(json.dumps({c["case_id"]: c["reported_variants"] for c in CASES}, indent=1))

## 2. Assess each reported variant

In [ ]:
lookup = {row["accession"]: row.asDict()
          for row in reference.collect()}

assessment_rows, coverage_rows, summary_rows = [], [], []

for case in CASES:
    case_id = case["case_id"]
    tiers = []

    for accession in case["reported_variants"]:
        record = lookup.get(accession)
        if record is None:
            assessment_rows.append({
                "run_id": RUN_ID, "case_id": case_id, "accession": accession,
                "gene_symbol": None, "variant_title": None, "hgvs_c": None,
                "protein_change": None, "clinical_significance": None,
                "review_status": None, "review_confidence": None,
                "condition_names": None, "condition_identifiers": None,
                "molecular_consequence": None,
                "assessment_state": "no_reference_entry",
                "reportable": False,
                "interpretation_note": ("No submitted assertion for this variant in the "
                                        "release read. Absence of a submission is not "
                                        "evidence that the variant is benign."),
                "generated_at_utc": GENERATED_AT,
            })
            tiers.append("no_reference_entry")
            continue

        significance = record["clinical_significance"]
        # Only Pathogenic / Likely pathogenic are actionable. Uncertain is reported but
        # explicitly not actionable; benign findings are recorded and not reported.
        reportable = significance in ("Pathogenic", "Likely pathogenic",
                                      "Uncertain significance", "Conflicting")
        if significance == "Uncertain significance":
            note = ("Variant of uncertain significance. Under ACMG/AMP guidance a VUS "
                    "should not be used to direct clinical management in either "
                    "direction, and is not a negative result.")
        elif significance == "Conflicting":
            note = ("Submitters disagree on this variant in the release read. The "
                    "conflict is itself the finding and is not resolved here.")
        elif significance in ("Benign", "Likely benign"):
            note = "Classified benign in the release read."
        else:
            note = ("Submitted assertion present. Review status determines how much "
                    "confidence the assertion carries.")

        assessment_rows.append({
            "run_id": RUN_ID, "case_id": case_id, "accession": accession,
            "gene_symbol": record["gene_symbol"],
            "variant_title": record["variant_title"],
            "hgvs_c": record["hgvs_c"],
            "protein_change": record["protein_change"],
            "clinical_significance": significance,
            "review_status": record["review_status"],
            "review_confidence": record["review_confidence"],
            "condition_names": record["condition_names"],
            "condition_identifiers": record["condition_identifiers"],
            "molecular_consequence": record["molecular_consequence"],
            "assessment_state": "classified",
            "reportable": bool(reportable),
            "interpretation_note": note,
            "generated_at_utc": GENERATED_AT,
        })
        tiers.append(significance or "unclassified")

    # Coverage is stated per case, whether or not anything was found.
    covered = [row["gene_symbol"] for row in
               lake_table("bronze_lakehouse", "bronze_reference_coverage")
               .filter((F.col("run_id") == RUN_ID) & (F.col("status") == "available"))
               .select("gene_symbol").collect()]
    uncalled = case["uncalled_gene"]
    coverage_rows.append({
        "run_id": RUN_ID, "case_id": case_id,
        "panel_applied": case["panel_applied"],
        "genes_covered": len(covered),
        "gene_of_interest_not_covered": uncalled,
        "coverage_state": "gene_not_covered" if uncalled else "panel_complete",
        "coverage_note": (
            f"{uncalled} is implicated by the referral indication but is not on "
            f"{case['panel_applied']}. No conclusion about {uncalled} can be drawn from "
            "this result." if uncalled else
            f"All {len(covered)} genes on {case['panel_applied']} returned reference data."),
        "generated_at_utc": GENERATED_AT,
    })

    order = ["Pathogenic", "Likely pathogenic", "Conflicting",
             "Uncertain significance", "no_reference_entry",
             "Likely benign", "Benign"]
    highest = next((t for t in order if t in tiers), None)
    summary_rows.append({
        "run_id": RUN_ID, "case_id": case_id,
        "referral_indication": case["referral_indication"],
        "phenotype_terms": case["phenotype_terms"],
        "panel_applied": case["panel_applied"],
        "variants_reported": len(case["reported_variants"]),
        "highest_tier": highest,
        "has_coverage_gap": bool(uncalled),
        "generated_at_utc": GENERATED_AT,
    })

print(f"assessments: {len(assessment_rows)}  cases: {len(summary_rows)}")

In [ ]:
ASSESSMENT_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("case_id", StringType(), True),
    StructField("accession", StringType(), True),
    StructField("gene_symbol", StringType(), True),
    StructField("variant_title", StringType(), True),
    StructField("hgvs_c", StringType(), True),
    StructField("protein_change", StringType(), True),
    StructField("clinical_significance", StringType(), True),
    StructField("review_status", StringType(), True),
    StructField("review_confidence", IntegerType(), True),
    StructField("condition_names", StringType(), True),
    StructField("condition_identifiers", StringType(), True),
    StructField("molecular_consequence", StringType(), True),
    StructField("assessment_state", StringType(), True),
    StructField("reportable", BooleanType(), True),
    StructField("interpretation_note", StringType(), True),
    StructField("generated_at_utc", StringType(), True),
])

COVERAGE_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("case_id", StringType(), True),
    StructField("panel_applied", StringType(), True),
    StructField("genes_covered", IntegerType(), True),
    StructField("gene_of_interest_not_covered", StringType(), True),
    StructField("coverage_state", StringType(), True),
    StructField("coverage_note", StringType(), True),
    StructField("generated_at_utc", StringType(), True),
])

SUMMARY_SCHEMA = StructType([
    StructField("run_id", StringType(), True),
    StructField("case_id", StringType(), True),
    StructField("referral_indication", StringType(), True),
    StructField("phenotype_terms", StringType(), True),
    StructField("panel_applied", StringType(), True),
    StructField("variants_reported", IntegerType(), True),
    StructField("highest_tier", StringType(), True),
    StructField("has_coverage_gap", BooleanType(), True),
    StructField("generated_at_utc", StringType(), True),
])


def write_run_scoped(dataframe, table_name):
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (dataframe.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING: {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:110]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (dataframe.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")


def frame(records, schema):
    columns = [field.name for field in schema.fields]
    return spark.createDataFrame(
        [tuple(record.get(column) for column in columns) for record in records],
        schema=schema)


write_run_scoped(frame(assessment_rows, ASSESSMENT_SCHEMA), "gold_case_variant_assessment")
write_run_scoped(frame(coverage_rows, COVERAGE_SCHEMA), "gold_case_coverage")
write_run_scoped(frame(summary_rows, SUMMARY_SCHEMA), "gold_case_summary")

In [ ]:
display(spark.read.table("gold_case_variant_assessment")
        .filter(F.col("run_id") == RUN_ID)
        .select("case_id", "gene_symbol", "accession", "clinical_significance",
                "assessment_state", "reportable", "review_status"))